In [1]:
from pathlib import Path

import numpy as np
from ls_mcmc import logging, sampling

from cardiac_electrophysiology import mcmc_builder, posterior_builder

In [2]:
posterior_settings = posterior_builder.PosteriorBuilderSettings(
    paths=posterior_builder.Paths(
        mesh_path=Path("../data/mesh.vtu"),
        basis_vecs_path=Path("../data/basis_vecs.npy"),
        prior_mean_path=Path("../data/prior_mean_from_sde.npy"),
        ground_truth_path=Path("../data/ground_truth_from_sde.npy"),
        log_file_path=Path("../results/lsbip_logfile.log"),
    ),
    prior_parameters=posterior_builder.PriorParameters(
        kappa=0.005,
        tau=100,
        seed=0,
    ),
    eikonal_parameters=posterior_builder.EikonalParameters(
        solver_tolerance=1e-6,
        max_num_iterations=1000,
        max_value=1000,
        initial_site_ind=12650,
        longitudinal_velocity=3,
        transversal_velocity=1,
    ),
    observation_parameters=posterior_builder.ObservationParameters(
        num_observations=1000,
        noise_variance=1e-4,
        seed=0,
    ),
    logger_settings=posterior_builder.LoggerSettings(
        do_printing=False,
        write_mode="w",
    ),
)
builder = posterior_builder.PosteriorBuilder(posterior_settings)
posterior, additional_output = builder.build(return_additional_data=True)

In [3]:
map_estimate = np.load("../results/260422_map_estimate.npy")
builder_settings = mcmc_builder.MCMCBuilderSettings(
    mcmc_model_settings=mcmc_builder.MCMCModelSettings(
        log_posterior=posterior,
        reference_point=map_estimate,
        step_width=1e-3,
        index_to_track=42,
    ),
    logging_settings=logging.LoggerSettings(
        do_printing=True,
        logfile_path=None,
    )
)
builder = mcmc_builder.MCMCBuilder(builder_settings)
mcmc_sampler = builder.build()

In [ ]:
rng = np.random.default_rng(seed=0)
noise = rng.normal(0, 1e-4, size=map_estimate.shape)
sampler_settings=sampling.SamplerRunSettings(
    num_samples=500,
    initial_state=additional_output.prior_mean_parameter,
    print_interval=1,
)
storage, outputs = mcmc_sampler.run(sampler_settings)

| Iteration   | Time        | Accept Rate    | Component 42   | Run_mean_C_42  | 
--------------------------------------------------------------------------------
| 0.000e+00   | 8.153e-02   | +1.000e+00     | -1.292e-01     | -1.292e-01     | 
| 1.000e+00   | 3.947e+00   | +1.000e+00     | -1.692e-01     | -1.492e-01     | 
| 2.000e+00   | 5.159e+00   | +6.667e-01     | -1.692e-01     | -1.559e-01     | 
| 3.000e+00   | 6.474e+00   | +5.000e-01     | -1.692e-01     | -1.592e-01     | 
| 4.000e+00   | 7.903e+00   | +4.000e-01     | -1.692e-01     | -1.612e-01     | 
| 5.000e+00   | 9.123e+00   | +3.333e-01     | -1.692e-01     | -1.626e-01     | 
| 6.000e+00   | 1.038e+01   | +4.286e-01     | -1.832e-01     | -1.655e-01     | 
| 7.000e+00   | 1.159e+01   | +3.750e-01     | -1.832e-01     | -1.677e-01     | 
| 8.000e+00   | 1.287e+01   | +3.333e-01     | -1.832e-01     | -1.694e-01     | 
| 9.000e+00   | 1.406e+01   | +3.000e-01     | -1.832e-01     | -1.708e-01     | 
| 1.000e+01   | 1